# Chapter 14 — cuBLAS for Matmul

> Course: **llm.c — Zero to Hero**, Chapter 14 of ~20.
> Builds on Chapter 4 (CPU matmul) and Chapters 9–13 (CUDA basics).

In Chapter 4 we saw that even a *good* CPU matmul (with OpenMP + register tiling) is roughly 2× slower than naive — and that's after substantial work. A *good GPU matmul* requires far more: shared-memory tiling, register tiling at the warp level, double-buffered loads, careful tile-size selection, tensor-core mixed-precision instructions on Hopper/Blackwell, etc. Production GPU matmul kernels are **tens of thousands of lines** and tuned for every architecture separately.

We're not going to write that. NVIDIA's [**cuBLAS**](https://docs.nvidia.com/cuda/cublas/) library has spent two decades being optimized by full-time experts. **It will run faster than anything you write.** `llm.c` calls cuBLAS for every matrix multiply. So will we.

This chapter is mostly about **how to use** cuBLAS effectively: the API, the row-major / column-major confusion, the `cublasLt` "epilogue" trick that fuses matmul with bias+activation, and the fact that the speed difference between matmul and a hand-rolled kernel is the entire reason GPU training is fast.

### Learning objectives

By the end of this chapter you will:

- Call `cublasGemmEx` to multiply two float matrices on the GPU.
- Understand the column-major convention and the `op_A` / `op_B` transpose flags.
- Read `llmc/matmul.cuh::matmul_cublaslt` and explain its arguments.
- Recognize the **fused epilogue** trick: matmul + bias + GELU in one kernel call.


## 1. Concept — Why cuBLAS

GPU matmul is a research field. The good kernels do:

- **Block-level tiling**: load `(BM, BK)` of A and `(BK, BN)` of B into shared memory, compute a `(BM, BN)` block of C from those tiles. Reuse each loaded element of A and B many times across threads.
- **Warp-level tiling**: each warp owns a sub-block of the output, accumulating in registers.
- **Tensor cores**: special FP16/BF16 matrix-multiply units that do 16x16x16 matrix multiplies in one instruction (one warp = one tensor core MMA).
- **Double-buffering / async copies**: while one tile is computing, the next tile is loading from global memory.
- **Tile-size selection per architecture**: `(BM, BN, BK) = (128, 256, 32)` may be optimal on H100 but bad on A100. cuBLAS picks the right one for you.

Writing this from scratch is months of work. cuBLAS does it; you call a function. Done.

`llm.c`'s `matmul_cublaslt` (in `llmc/matmul.cuh`) is ~80 lines and wraps `cublasLtMatmul`. That is the entirety of GPT-2's matmul code on GPU.


## 2. Concept — Column-Major Convention

cuBLAS is descended from FORTRAN BLAS, which uses **column-major** storage:

> `A[i, j]` lives at memory offset `i + j * lda`, where `lda` is the row stride (column count of the *transposed* view).

PyTorch and `llm.c` use **row-major** (`A[i, j]` at offset `i * stride + j`). Calling cuBLAS from row-major code requires a small mental gymnastic.

The trick: **column-major-A is row-major-A-transposed**. A matrix `A` of shape `(M, N)` in row-major looks like a `(N, M)` matrix in column-major. So if you want `C = A @ B` in row-major, you can ask cuBLAS to compute `C.T = B.T @ A.T` in column-major, which uses the same bytes. cuBLAS sees the row-major `A` as col-major `A.T`, and so on.

In practice you call `cublasGemmEx` with the operands swapped: `cublasGemmEx(C = B @ A, ...)` and pass `op_A = N`, `op_B = N` — and you get the row-major answer because the bytes line up.

This trick — *swap A and B, leave the rest untouched* — is what every framework that wraps cuBLAS does. We'll see it in `matmul_cublaslt` below.


## 3. Demo — Calling cuBLAS

In [ ]:
!mkdir -p course/ch14_build


In [ ]:
%%writefile course/ch14_build/cublas_demo.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

// CPU reference: C = A @ B with A (M, K), B (K, N), C (M, N), all row-major
void matmul_cpu(float* C, const float* A, const float* B, int M, int N, int K) {
    for (int m = 0; m < M; m++)
        for (int n = 0; n < N; n++) {
            float s = 0;
            for (int k = 0; k < K; k++) s += A[m*K + k] * B[k*N + n];
            C[m*N + n] = s;
        }
}

int main(void) {
    int M = 64, K = 128, N = 96;

    float *h_A = (float*) malloc(M*K*4);
    float *h_B = (float*) malloc(K*N*4);
    float *h_C_ref = (float*) malloc(M*N*4);
    float *h_C_gpu = (float*) malloc(M*N*4);
    for (int i = 0; i < M*K; i++) h_A[i] = (float)((i*7) % 13) / 5.0f - 1.0f;
    for (int i = 0; i < K*N; i++) h_B[i] = (float)((i*11) % 17) / 7.0f - 1.0f;
    matmul_cpu(h_C_ref, h_A, h_B, M, N, K);

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, M*K*4); cudaMalloc(&d_B, K*N*4); cudaMalloc(&d_C, M*N*4);
    cudaMemcpy(d_A, h_A, M*K*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, K*N*4, cudaMemcpyHostToDevice);

    cublasHandle_t handle;
    cublasCreate(&handle);

    // Row-major C = A @ B
    // Column-major view: C^T = B^T @ A^T. We ask cuBLAS to compute (B^T @ A^T) into C in col-major.
    //
    // cublasSgemm signature (column-major):
    //   C(M', N') = alpha * op(A')(M', K') * op(B')(K', N') + beta * C(M', N')
    //
    // We want output C with M rows, N cols (row-major). In col-major those are (N, M).
    // First operand should be B (col-major B is (N, K)), second should be A (col-major A is (K, M)).
    // No transposes needed. M' = N, N' = M, K' = K.
    float alpha = 1.0f, beta = 0.0f;
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                N, M, K,                 // M', N', K' in column-major
                &alpha,
                d_B, N,                  // first operand: B (col-major view, leading dim N)
                d_A, K,                  // second operand: A (col-major view, leading dim K)
                &beta,
                d_C, N);                 // output C (col-major leading dim N)

    cudaMemcpy(h_C_gpu, d_C, M*N*4, cudaMemcpyDeviceToHost);

    float maxerr = 0;
    for (int i = 0; i < M*N; i++) {
        float e = fabsf(h_C_gpu[i] - h_C_ref[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("cuBLAS Sgemm  M=%d N=%d K=%d  max diff vs CPU = %.2e\n", M, N, K, maxerr);

    cublasDestroy(handle);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C_ref); free(h_C_gpu);
    return 0;
}


In [ ]:
!nvcc -O2 -lcublas -o course/ch14_build/cublas_demo course/ch14_build/cublas_demo.cu && ./course/ch14_build/cublas_demo


You should see ~`1e-4` agreement (cuBLAS may use TF32 or different summation order than scalar CPU). The argument order for the row-major-using-column-major trick is the part that takes getting used to — in practice you write a small wrapper once and never think about it again.


## 4. The `matmul_cublaslt` Wrapper in `llm.c`

`llm.c`'s production code uses `cublasLt` (the "lighter-weight, more configurable" cuBLAS API), which exposes more knobs but lets us **fuse a bias add and even a GELU activation into the matmul itself**. Here's the key snippet from [`llmc/matmul.cuh`](llmc/matmul.cuh):

```cpp
void matmul_cublaslt(floatX* d, const floatX* a, const floatX* b, const floatX* bias,
                     int m, int n, int k, cudaStream_t stream=0,
                     bool transA=true, bool transB=false,
                     int batch_count=0, size_t strideA=0, size_t strideB=0, size_t strideOut=0,
                     bool accumulate=false, floatX* pre_gelu=NULL, bool backward=false) {
    // ... lots of setup ...

    cublasLtMatmulDescCreate(&operationDesc, ...);

    // Set epilogue: which op happens AFTER the matmul, fused into the same kernel?
    cublasLtEpilogue_t epilogue;
    if (bias != NULL && pre_gelu != NULL) {
        epilogue = backward ? CUBLASLT_EPILOGUE_DGELU_BGRADB
                            : CUBLASLT_EPILOGUE_GELU_AUX_BIAS;       // matmul + bias + GELU, save pre-GELU
    } else if (bias != NULL) {
        epilogue = backward ? CUBLASLT_EPILOGUE_BGRADB : CUBLASLT_EPILOGUE_BIAS;
    } else if (pre_gelu != NULL) {
        epilogue = backward ? CUBLASLT_EPILOGUE_DGELU : CUBLASLT_EPILOGUE_GELU_AUX;
    } else {
        epilogue = CUBLASLT_EPILOGUE_DEFAULT;
    }
    cublasLtMatmulDescSetAttribute(operationDesc, CUBLASLT_MATMUL_DESC_EPILOGUE, &epilogue, sizeof(epilogue));

    // ... layout descriptors, algorithm heuristic ...

    cublasLtMatmul(handle, operationDesc, &alpha, a, ALayout, b, BLayout, &beta, d, CLayout, d, CLayout,
                   &heuristic.algo, NULL, 0, stream);
}
```

The interesting bit is **`epilogue`**. Without fusion, the FFN block requires three kernel launches:

```
matmul:   y = x @ W1.T + 0
bias add: y = y + b1
gelu:     z = gelu(y)
```

With cuBLASLt's `CUBLASLT_EPILOGUE_GELU_AUX_BIAS` epilogue, all three happen **inside the same matmul kernel**:

```
matmul + bias + gelu:  z = gelu(x @ W1.T + b1)   (one launch, one pass over global memory)
```

The "AUX" suffix means cuBLASLt also writes the **pre-activation** result (`y` before GELU) to a buffer — needed for the backward pass, where you need both the input to GELU and the output. We saw the `fch` (pre-GELU) and `fch_gelu` (post-GELU) tensors in `ActivationTensors` (Chapter 8); `pre_gelu` here is `fch`.

Same idea for backward: `CUBLASLT_EPILOGUE_DGELU_BGRADB` does **GELU backward + matmul + bias gradient** in one kernel.


## 5. The Productivity Story

Take a moment to appreciate what this means for the codebase.

- The CPU `matmul_forward + matmul_forward_naive` from Chapter 4: ~70 lines, ~150 GFLOPs/s on the workstation CPU.
- The GPU `matmul_cublaslt` wrapper: ~80 lines, ~30+ TFLOP/s on the 4080.
- That's a ~200× speedup, achieved by *delegating to a library*.

Other GPU layers in `llm.c` (LayerNorm, attention, GELU, fused classifier) are hand-rolled because they're either bandwidth-bound (a library couldn't help) or have a memory layout that no library covers. Matmul is where libraries win unconditionally.

This is why ~95% of the FLOPs in a Transformer training step are inside `cublasLtMatmul` calls. The rest of `llm.c`'s CUDA code orchestrates: precondition the data, call the matmul, postprocess the result.


## 6. Translation Bridge

| What it does | How |
|---|---|
| `nn.Linear(C, OC).forward(x)` in PyTorch | Calls cuBLAS internally |
| Hand-roll a GPU matmul | Don't. Even tuned ones are slower than cuBLAS. |
| `(C, OC) @ (OC, C)` row-major | `cublasSgemm(N, N, N=OC, M=BT, K=C, ...)` with operands swapped |
| Matmul + bias + activation | `cublasLtMatmul` with `EPILOGUE_GELU_AUX_BIAS` |
| Mixed-precision matmul | `cublasGemmEx` with `CUDA_R_16BF` data type, `CUDA_R_32F` compute type, `CUBLAS_COMPUTE_32F` |

The general rule: **if your operation is dense linear algebra, call cuBLAS.** If it's anything else (reductions, softmax, layer-specific math), hand-roll a kernel.


## 7. TODO Exercise — Sgemm Wrapper for Row-Major

Write a wrapper `row_major_gemm(C, A, B, M, N, K)` that does row-major `C = A @ B` using `cublasSgemm`. Verify against a CPU implementation.


In [ ]:
%%writefile course/ch14_build/exercise1.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

void matmul_cpu(float* C, const float* A, const float* B, int M, int N, int K) {
    for (int m = 0; m < M; m++)
        for (int n = 0; n < N; n++) {
            float s = 0;
            for (int k = 0; k < K; k++) s += A[m*K + k] * B[k*N + n];
            C[m*N + n] = s;
        }
}

// TODO: write a wrapper that does row-major C = A @ B using cublasSgemm.
// Hint: in column-major land, ask cuBLAS to do C^T = B^T @ A^T (no actual transposes).
//       Pass B first, A second, with leading dims N and K respectively.
void row_major_gemm(cublasHandle_t handle, float* C, const float* A, const float* B,
                    int M, int N, int K) {
    // float alpha = 1.0f, beta = 0.0f;
    // TODO: cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, B, N, A, K, &beta, C, N);
}

int main(void) {
    int M = 32, K = 64, N = 48;
    float *h_A = (float*) malloc(M*K*4);
    float *h_B = (float*) malloc(K*N*4);
    float *h_C_ref = (float*) malloc(M*N*4);
    float *h_C_gpu = (float*) malloc(M*N*4);
    for (int i = 0; i < M*K; i++) h_A[i] = (float)(i % 7) / 3.0f - 1.0f;
    for (int i = 0; i < K*N; i++) h_B[i] = (float)(i % 11) / 5.0f - 1.0f;
    matmul_cpu(h_C_ref, h_A, h_B, M, N, K);

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, M*K*4); cudaMalloc(&d_B, K*N*4); cudaMalloc(&d_C, M*N*4);
    cudaMemcpy(d_A, h_A, M*K*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, K*N*4, cudaMemcpyHostToDevice);

    cublasHandle_t handle; cublasCreate(&handle);
    row_major_gemm(handle, d_C, d_A, d_B, M, N, K);
    cudaMemcpy(h_C_gpu, d_C, M*N*4, cudaMemcpyDeviceToHost);

    float maxerr = 0;
    for (int i = 0; i < M*N; i++) {
        float e = fabsf(h_C_gpu[i] - h_C_ref[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("max diff: %.2e %s\n", maxerr, maxerr < 1e-3 ? "PASS" : "FAIL");
    cublasDestroy(handle);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C_ref); free(h_C_gpu);
    return 0;
}


In [ ]:
!nvcc -O2 -lcublas -o course/ch14_build/exercise1 course/ch14_build/exercise1.cu && ./course/ch14_build/exercise1


### Solution

In [ ]:
%%writefile course/ch14_build/exercise1_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

void matmul_cpu(float* C, const float* A, const float* B, int M, int N, int K) {
    for (int m = 0; m < M; m++) for (int n = 0; n < N; n++) {
        float s = 0; for (int k = 0; k < K; k++) s += A[m*K + k] * B[k*N + n];
        C[m*N + n] = s;
    }
}

void row_major_gemm(cublasHandle_t handle, float* C, const float* A, const float* B,
                    int M, int N, int K) {
    float alpha = 1.0f, beta = 0.0f;
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K,
                &alpha, B, N, A, K, &beta, C, N);
}

int main(void) {
    int M = 32, K = 64, N = 48;
    float *h_A = (float*) malloc(M*K*4);
    float *h_B = (float*) malloc(K*N*4);
    float *h_C_ref = (float*) malloc(M*N*4);
    float *h_C_gpu = (float*) malloc(M*N*4);
    for (int i = 0; i < M*K; i++) h_A[i] = (float)(i % 7) / 3.0f - 1.0f;
    for (int i = 0; i < K*N; i++) h_B[i] = (float)(i % 11) / 5.0f - 1.0f;
    matmul_cpu(h_C_ref, h_A, h_B, M, N, K);
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, M*K*4); cudaMalloc(&d_B, K*N*4); cudaMalloc(&d_C, M*N*4);
    cudaMemcpy(d_A, h_A, M*K*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, K*N*4, cudaMemcpyHostToDevice);
    cublasHandle_t handle; cublasCreate(&handle);
    row_major_gemm(handle, d_C, d_A, d_B, M, N, K);
    cudaMemcpy(h_C_gpu, d_C, M*N*4, cudaMemcpyDeviceToHost);
    float maxerr = 0;
    for (int i = 0; i < M*N; i++) {
        float e = fabsf(h_C_gpu[i] - h_C_ref[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("max diff: %.2e %s\n", maxerr, maxerr < 1e-3 ? "PASS" : "FAIL");
    cublasDestroy(handle);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C_ref); free(h_C_gpu);
    return 0;
}


In [ ]:
!nvcc -O2 -lcublas -o course/ch14_build/exercise1_sol course/ch14_build/exercise1_sol.cu && ./course/ch14_build/exercise1_sol


## Further Reading

**Source of truth**

- [cuBLAS Library documentation](https://docs.nvidia.com/cuda/cublas/) — API reference for `cublasGemmEx` / `cublasSgemm`, the column-major convention, and the `op_A` / `op_B` transpose flags.
- `llmc/matmul.cuh` (`matmul_cublaslt`) in this repo — the production wrapper, including the fused bias+GELU **cuBLASLt epilogue**.

**Going deeper**

- [CUDA C++ Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html) — the "prefer existing libraries" guidance and the arithmetic-intensity reasoning behind why cuBLAS wins.


## Recap

You now know:

- **cuBLAS** is NVIDIA's hand-tuned linear-algebra library. Use it for matmul; never write your own.
- **Row-major ↔ column-major** trick: pass B first, A second, no transposes, leading dims `N` and `K` — get back row-major `C = A @ B`.
- **`cublasLt` epilogues** fuse bias and GELU into the matmul kernel (`CUBLASLT_EPILOGUE_GELU_AUX_BIAS`).
- ~95% of GPT-2 training FLOPs happen inside cuBLAS calls.

### Going deeper (optional)

**Chapter 14a — GPU Matmul Deep Dive.** Want to know *what cuBLAS is actually doing* inside, why a naive kernel is slow, and how tiling / tensor cores / fused epilogues / batched GEMM work? The companion notebook `Chapter_14a_GPU_Matmul_Deep_Dive.ipynb` walks through it with figures and a live benchmark.

### What's next

**Chapter 15 — Kernel Fusion.** Beyond matmul, hand-rolled kernels also benefit from fusion. We'll meet `fused_classifier.cuh` (softmax + cross-entropy + first backward step in one pass), `fused_residual_forward.cu` (residual + layernorm), and the general principle: **fewer trips to global memory = more speed**.